# Baseline

Создадим простейшую модель из коробки. Она будет служить **нижней границей качества**.

Baseline-модель: Ridge is `scikit-learn`.

Метрики:
- **RMSE** (основная) — корень из среднеквадратичной ошибки в логарифмическом пространстве, чувствителен к выбросам.  
- **MAE** — средняя абсолютная ошибка в логарифмах, интерпретируема.  
- **R²** — доля объяснённой дисперсии.

**План:**
1. Загрузить обработанные данные (`data/processed/eda_processed.csv`).
2. Разделить на train / validation / test (с фиксированным seed для воспроизводимости).
3. Закодировать категориальные признаки и `developer_name`.
4. Обучить Ridge на тренировочной выборке, оценить на валидационной.
5. Зафиксировать результаты для сравнения с будущими экспериментами.

## Загрузка данных

Загрузим данные, подготовленные в файле `01_eda.ipynb`.

In [91]:
import pandas as pd

df = pd.read_csv("../data/processed/eda_processed.csv")

Убедимся, что логарифмированный таргет на месте.

In [92]:
assert 'log_total_revenue' in df.columns, "Нет log_total_revenue!"

print("Датасет загружен, форма:", df.shape)
df.head()

Датасет загружен, форма: (9980, 34)


,developer_name,genre,sub_genre,monetization_model,contains_ads,has_in_app_purchases,multiplayer_support,age_rating,art_style,engine_used,...,had_soft_launch,days_soft_to_release,log_total_revenue,price_usd_raw,marketing_spend_usd_raw,apk_size_mb_raw,wishlist_prelaunch_raw,release_year,release_month,release_dayofweek
0,Studio_205,Casual,Party Game,Freemium,True,True,Unknown,Mature 17+,2D Cartoon,Unity,...,True,40,9.654661,0.00,4851.05,244.04,644,2025,5,4
1,Studio_923,RPG,Idle RPG,Free,True,True,Online,Teen,Minimalist,Unity,...,True,158,10.248295,0.00,4143.82,46.46,1463,2019,5,1
2,Studio_921,RPG,Idle RPG,Free,True,True,Unknown,Everyone,Anime,Unity,...,True,31,8.642447,0.00,2539.93,236.90,328,2018,12,5
3,Studio_1873,Casual,Idle Clicker,Paid,True,True,Unknown,Everyone,Minimalist,Unity,...,True,45,13.556460,1.03,114491.05,51.11,3269,2019,8,1
4,Studio_1608,RPG,MMORPG,Free,True,True,Unknown,Teen,3D Realistic,Unity,...,True,143,11.908584,0.00,9783.61,241.94,8172,2025,12,6


## Разделение данных

Разделим данные на три части: train (70%), validation (15%), test (15%). 

Учитываем, что датасет синтетический и не привязан к реальному времени, поэтому используем случайное разделение. Проверим, чтобы разработчики (developer_name) не пересекались между трейном и тестом, что могло бы привести к неявной утечке.

Зафиксируем `RANDOM_STATE` для воспроизводимости.

In [93]:
from sklearn.model_selection import train_test_split

RANDOM_STATE = 42

TEST_SIZE = 0.15
VALIDATION_SIZE = 0.15

X = df.drop(columns=['total_revenue_usd', 'log_total_revenue'])
y = df['log_total_revenue']

Первый сплит: отделяем test (15%).

In [94]:
X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

Второй сплит: выделяем validation (от исходного 15% => из оставшегося 0.15/0.85).

In [95]:
val_size = VALIDATION_SIZE / (1 - TEST_SIZE)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=val_size, random_state=RANDOM_STATE
)

In [96]:
print(f"Train: {X_train.shape[0]} ({X_train.shape[0]/len(df):.1%})")
print(f"Validation: {X_val.shape[0]} ({X_val.shape[0]/len(df):.1%})")
print(f"Test: {X_test.shape[0]} ({X_test.shape[0]/len(df):.1%})")

Train: 6986 (70.0%)
Validation: 1497 (15.0%)
Test: 1497 (15.0%)


## Оптимизация

На предыдущем шаге мы оставили переменную `developer_name`. Посмотрим на пересечения:

In [97]:
train_devs = set(X_train['developer_name'].unique())
val_devs = set(X_val['developer_name'].unique())
test_devs = set(X_test['developer_name'].unique())
print("Пересечение разработчиков train/val:", len(train_devs & val_devs))
print("Пересечение разработчиков train/test:", len(train_devs & test_devs))
print("Пересечение разработчиков val/test:", len(val_devs & test_devs))

Пересечение разработчиков train/val: 1029
Пересечение разработчиков train/test: 1020
Пересечение разработчиков val/test: 532


Разработчики случайно распределены по выборкам, так как датасет синтетический.
Вследствие этого имеются пересечения.

Для baseline-модели **удалим** эту переменную.
В экспериментах вернём и посмотрим, даст ли это какой-то прирост.

In [98]:
X_train = X_train.drop(columns=['developer_name'])
X_val = X_val.drop(columns=['developer_name'])
X_test = X_test.drop(columns=['developer_name'])

## Кодирование признаков

Категориальные признаки (тип `category`) и булевы флаги кодируем через One-Hot.

Числовые признаки оставляем без изменений (уже логарифмированы).

In [99]:
from sklearn.preprocessing import OneHotEncoder

cat_features = X_train.select_dtypes(include=['str', 'bool']).columns.tolist()
print("Категориальные признаки для One-Hot:", cat_features)

Категориальные признаки для One-Hot: ['genre', 'sub_genre', 'monetization_model', 'contains_ads', 'has_in_app_purchases', 'multiplayer_support', 'age_rating', 'art_style', 'engine_used', 'region_focus', 'publisher_tier', 'was_featured_on_store', 'is_weekend_release', 'has_seasonal_event', 'cross_platform_available', 'cloud_save_support', 'country_code_primary', 'had_soft_launch']


Используем One-Hot Encoding с обработкой неизвестных категорий.

In [100]:
ohe = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
ohe.fit(X_train[cat_features])

def encode(df, encoder, cat_cols):
    # Преобразуем категории в матрицу 0/1
    encoded = encoder.transform(df[cat_cols])
    # Получаем имена новых столбцов
    encoded_cols = encoder.get_feature_names_out(cat_cols)
    # Оборачиваем в DataFrame
    encoded_df = pd.DataFrame(encoded, index=df.index, columns=encoded_cols)
    # Удаляем исходные категории и присоединяем закодированные
    df = df.drop(columns=cat_cols)
    # Приклеиваем новые столбцы справа
    return pd.concat([df, encoded_df], axis=1)

Теперь кодируем все выборки, оставляя только числа.

In [101]:
X_train_encoded = encode(X_train, ohe, cat_features)
X_val_encoded = encode(X_val, ohe, cat_features)
X_test_encoded = encode(X_test, ohe, cat_features)

print("Размер после One-Hot:", X_train_encoded.shape)

Размер после One-Hot: (6986, 99)


## Обучение

In [102]:
import numpy as np
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

baseline_model = Ridge(alpha=1.0, random_state=RANDOM_STATE)
baseline_model.fit(X_train_encoded, y_train)

,"alpha alpha: {float, ndarray of shape (n_targets,)}, default=1.0Constant that multiplies the L2 term, controlling regularizationstrength. `alpha` must be a non-negative float i.e. in `[0, inf)`.When `alpha = 0`, the objective is equivalent to ordinary leastsquares, solved by the :class:`LinearRegression` object. For numericalreasons, using `alpha = 0` with the `Ridge` object is not advised.Instead, you should use the :class:`LinearRegression` object.If an array is passed, penalties are assumed to be specific to thetargets. Hence they must correspond in number.",1.0
,"fit_intercept fit_intercept: bool, default=TrueWhether to fit the intercept for this model. If setto false, no intercept will be used in calculations(i.e. ``X`` and ``y`` are expected to be centered).",True
,"copy_X copy_X: bool, default=TrueIf True, X will be copied; else, it may be overwritten.",True
,"max_iter max_iter: int, default=NoneMaximum number of iterations for conjugate gradient solver.For 'sparse_cg' and 'lsqr' solvers, the default value is determinedby scipy.sparse.linalg. For 'sag' solver, the default value is 1000.For 'lbfgs' solver, the default value is 15000.",None
,"tol tol: float, default=1e-4The precision of the solution (`coef_`) is determined by `tol` whichspecifies a different convergence criterion for each solver:- 'svd': `tol` has no impact.- 'cholesky': `tol` has no impact.- 'sparse_cg': norm of residuals smaller than `tol`.- 'lsqr': `tol` is set as atol and btol of scipy.sparse.linalg.lsqr, which control the norm of the residual vector in terms of the norms of matrix and coefficients.- 'sag' and 'saga': relative change of coef smaller than `tol`.- 'lbfgs': maximum of the absolute (projected) gradient=max|residuals| smaller than `tol`... versionchanged:: 1.2 Default value changed from 1e-3 to 1e-4 for consistency with other linear models.",0.0001
,"solver solver: {'auto', 'svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'}, default='auto'Solver to use in the computational routines:- 'auto' chooses the solver automatically based on the type of data.- 'svd' uses a Singular Value Decomposition of X to compute the Ridge coefficients. It is the most stable solver, in particular more stable for singular matrices than 'cholesky' at the cost of being slower.- 'cholesky' uses the standard :func:`scipy.linalg.solve` function to obtain a closed-form solution.- 'sparse_cg' uses the conjugate gradient solver as found in :func:`scipy.sparse.linalg.cg`. As an iterative algorithm, this solver is more appropriate than 'cholesky' for large-scale data (possibility to set `tol` and `max_iter`).- 'lsqr' uses the dedicated regularized least-squares routine :func:`scipy.sparse.linalg.lsqr`. It is the fastest and uses an iterative procedure.- 'sag' uses a Stochastic Average Gradient descent, and 'saga' uses its improved, unbiased version named SAGA. Both methods also use an iterative procedure, and are often faster than other solvers when both n_samples and n_features are large. Note that 'sag' and 'saga' fast convergence is only guaranteed on features with approximately the same scale. You can preprocess the data with a scaler from :mod:`sklearn.preprocessing`.- 'lbfgs' uses L-BFGS-B algorithm implemented in :func:`scipy.optimize.minimize`. It can be used only when `positive` is True.All solvers except 'svd' support both dense and sparse data. However, only'lsqr', 'sag', 'sparse_cg', and 'lbfgs' support sparse input when`fit_intercept` is True... versionadded:: 0.17 Stochastic Average Gradient descent solver... versionadded:: 0.19 SAGA solver.",'auto'
,"positive positive: bool, default=FalseWhen set to ``True``, forces the coefficients to be positive.Only 'lbfgs' solver is supported in this case.",False
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag' or 'saga' to shuffle the data.See :term:`Glossary ` for details... versionadded:: 0.17 `random_state` to support Stochastic Average Gradient.",42


Предсказания на валидации.

In [103]:
y_val_pred = baseline_model.predict(X_val_encoded)

Соберём метрики.

In [104]:
rmse = np.sqrt(mean_squared_error(y_val, y_val_pred))
mae = mean_absolute_error(y_val, y_val_pred)
r2 = r2_score(y_val, y_val_pred)

Зафиксируем результат.

In [105]:
print("Baseline (Ridge) на валидации:")
print(f"  RMSE: {rmse:.4f}")
print(f"  MAE:  {mae:.4f}")
print(f"  R²:   {r2:.4f}")

Baseline (Ridge) на валидации:
  RMSE: 1.3366
  MAE:  1.0160
  R²:   0.5517


## Итог

Создана Baseline-модель (Ridge, alpha=1.0) на с One‑Hot кодированием.

Результат:
* RMSE: 1.3366
* MAE:  1.0160
* R²:   0.5517